# TLNS Proper Surrogate Test on Thornburg 4DWCM

Redesigned from the previous broken attempt. Key changes:

1. **Δy prediction** instead of absolute y(t+Δt). Fixes scale blowup.
2. **Cytoplasm only** (`_c` metabolites). Removes boundary flux.
3. **Δt = 600 s** (10 min). Enough time for real dynamics.
4. **77-dim null-space laws**, not just 10 element laws. More invariants enforced.
5. **Four-way comparison**: Identity / Ridge / MLP / TLNS — all predicting Δy.
6. **Three success criteria**: R² > 0.3 / conservation 100× better / runtime 1000× faster.

Expected runtime: ~15 min on Colab CPU (most of that is extracting the tarball).

## 1. Mount Drive and extract tarball to local disk

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess, time
from pathlib import Path

TARBALL = '/content/drive/MyDrive/Luthey-Schulten-Lab-Minimal_Cell-db048ac/MinCell_counts_and_fluxes.tar.gz'
LOCAL_EXTRACT = Path('/content/thornburg_csvs')

assert Path(TARBALL).exists(), f'Tarball not found at {TARBALL}'

# Check if already extracted
existing_csvs = list(LOCAL_EXTRACT.rglob('counts_and_fluxes.*.csv')) if LOCAL_EXTRACT.exists() else []
if len(existing_csvs) >= 10:
    print(f'Already extracted: {len(existing_csvs)} CSVs')
else:
    LOCAL_EXTRACT.mkdir(exist_ok=True)
    print(f'Extracting {TARBALL} to {LOCAL_EXTRACT}...')
    t0 = time.perf_counter()
    subprocess.run(['tar', '-xzf', TARBALL, '-C', str(LOCAL_EXTRACT)], check=True)
    print(f'Done in {time.perf_counter() - t0:.1f}s')

csv_files = sorted(LOCAL_EXTRACT.rglob('counts_and_fluxes.*.csv'),
                   key=lambda p: int(p.stem.split('.')[1]))
CSV_DIR = csv_files[0].parent
print(f'\nFound {len(csv_files)} CSVs in {CSV_DIR}')
print(f'Total size: {sum(f.stat().st_size for f in csv_files)/1e9:.1f} GB')

## 2. Load cytoplasm metabolites from all 50 cells

Only the `M_*_c` rows. Drops extracellular species (`_e`) and non-metabolite state variables. Expected: ~150 cytoplasm metabolites × 7200 time points × 50 cells.

In [ ]:
import numpy as np
import pandas as pd
import time

N_CELLS = 50  # use all 50

# Load cell 1 first to get metabolite list
t0 = time.perf_counter()
df = pd.read_csv(csv_files[0], low_memory=False)
species_col = df.columns[0]
species = df[species_col].values

# Cytoplasm metabolites only
is_cyto_met = np.array([isinstance(s, str) and s.startswith('M_') and s.endswith('_c')
                         for s in species])
MET_IDS = species[is_cyto_met]
print(f'Cytoplasm metabolites: {len(MET_IDS)}')
print(f'First 10: {list(MET_IDS[:10])}')
print(f'Loading cell 1 took {time.perf_counter()-t0:.1f}s')

# Extract just the cytoplasm rows from cell 1 data
time_cols = [c for c in df.columns if c != species_col]
n_time = len(time_cols)
print(f'Time points per cell: {n_time}')

# Load all cells: shape (n_cells, n_time, n_metabolites)
all_data = np.zeros((N_CELLS, n_time, len(MET_IDS)), dtype=np.float32)

print(f'\nLoading {N_CELLS} cells...')
t0 = time.perf_counter()
for i, csv_path in enumerate(csv_files[:N_CELLS]):
    df = pd.read_csv(csv_path, low_memory=False)
    # Confirm species match cell 1
    this_species = df[species_col].values
    this_cyto = np.array([isinstance(s, str) and s.startswith('M_') and s.endswith('_c')
                           for s in this_species])
    assert np.array_equal(this_species[this_cyto], MET_IDS), f'cell {i+1} has different metabolites'
    # Get cytoplasm data, transpose to (n_time, n_metabolites)
    cyto_df = df[this_cyto]
    all_data[i] = cyto_df[time_cols].values.astype(np.float32).T
    if (i+1) % 10 == 0:
        print(f'  {i+1}/{N_CELLS} cells loaded, elapsed {time.perf_counter()-t0:.1f}s')

print(f'\nAll data loaded: shape {all_data.shape}')
print(f'Memory: {all_data.nbytes/1e6:.0f} MB')
print(f'Range: [{all_data.min():.1f}, {all_data.max():.1f}]')
print(f'Median per-cell per-metabolite: {np.median(all_data):.1f}')

## 3. Parse iMB155 SBML and compute full 77-dim null-space conservation

The null space of the stoichiometry matrix `S` contains every vector `c` such that `c·Δy = 0` for any internal reaction flux. These are the linear conservation laws — all of them, not just the element-balance ones.

In [ ]:
import requests
from xml.etree import ElementTree as ET

SBML_URL = 'https://raw.githubusercontent.com/Luthey-Schulten-Lab/Minimal_Cell/master/CME_ODE/model_data/iMB155_NoH2O.xml'
# Try master branch first, fall back to main
try:
    r = requests.get(SBML_URL, timeout=30)
    r.raise_for_status()
except Exception:
    SBML_URL = 'https://raw.githubusercontent.com/Luthey-Schulten-Lab/Minimal_Cell/main/CME_ODE/model_data/iMB155_NoH2O.xml'
    r = requests.get(SBML_URL, timeout=30)
    r.raise_for_status()
sbml_text = r.text
print(f'Downloaded iMB155 SBML: {len(sbml_text):,} chars')

NS = {'sbml': 'http://www.sbml.org/sbml/level3/version1/core',
      'fbc':  'http://www.sbml.org/sbml/level3/version1/fbc/version2'}
root = ET.fromstring(sbml_text)
model = root.find('sbml:model', NS)

imb_mets = [sp.get('id') for sp in model.find('sbml:listOfSpecies', NS).findall('sbml:species', NS)]
imb_met_idx = {m: i for i, m in enumerate(imb_mets)}

reactions = model.find('sbml:listOfReactions', NS).findall('sbml:reaction', NS)
n_m, n_r = len(imb_mets), len(reactions)
S_full = np.zeros((n_m, n_r), dtype=np.float64)
for j, rx in enumerate(reactions):
    for side, sign in [('sbml:listOfReactants', -1), ('sbml:listOfProducts', +1)]:
        sec = rx.find(side, NS)
        if sec is not None:
            for ref in sec.findall('sbml:speciesReference', NS):
                S_full[imb_met_idx[ref.get('species')], j] += sign * float(ref.get('stoichiometry', 1.0))

print(f'iMB155: {n_m} metabolites × {n_r} reactions')

# Match Thornburg cytoplasm metabolites to iMB155 row indices
thornburg_in_imb = {}  # thornburg_idx -> imb155_idx
for t_i, mid in enumerate(MET_IDS):
    if mid in imb_met_idx:
        thornburg_in_imb[t_i] = imb_met_idx[mid]

print(f'Thornburg cytoplasm metabolites matching iMB155: {len(thornburg_in_imb)}/{len(MET_IDS)}')

# Restrict stoichiometry to matched rows only
matched_imb_rows = [thornburg_in_imb[i] for i in range(len(MET_IDS)) if i in thornburg_in_imb]
matched_thornburg_idx = [i for i in range(len(MET_IDS)) if i in thornburg_in_imb]
S_matched = S_full[matched_imb_rows]  # (n_matched, n_rxns)

# Drop reactions that involve unmatched metabolites
# (a reaction is valid for the subset iff all its participants are in the matched set)
matched_set = set(matched_imb_rows)
valid_rxn_mask = np.ones(n_r, dtype=bool)
for j in range(n_r):
    participants = np.where(S_full[:, j] != 0)[0]
    if not all(p in matched_set for p in participants):
        valid_rxn_mask[j] = False

S_subset = S_matched[:, valid_rxn_mask]
print(f'Subset: {S_subset.shape[0]} metabolites × {S_subset.shape[1]} reactions (dropped {(~valid_rxn_mask).sum()} rxns with unmatched mets)')

# Compute null space of S^T: all c such that c^T S = 0
U, sing, Vt = np.linalg.svd(S_subset.T, full_matrices=True)
tol = max(S_subset.shape) * np.finfo(float).eps * sing.max() if sing.size > 0 else 0
rank_S = int((sing > tol).sum())
n_laws = S_subset.shape[0] - rank_S
C_null = Vt[rank_S:]  # (n_laws, n_matched_metabolites)
print(f'Rank of S subset: {rank_S}')
print(f'Number of conservation laws: {n_laws}')

# Verify: C_null @ S_subset should be ~0
err = np.linalg.norm(C_null @ S_subset)
print(f'||C_null @ S_subset|| = {err:.2e} (should be ~0)')

# Build full C for all Thornburg metabolites (zero on unmatched)
C_full = np.zeros((n_laws, len(MET_IDS)), dtype=np.float64)
for col_in_subset, thorn_idx in enumerate(matched_thornburg_idx):
    C_full[:, thorn_idx] = C_null[:, col_in_subset]

print(f'\nFull conservation matrix (including zeros for unmatched mets): {C_full.shape}')

## 4. Measure how well each law actually holds in Thornburg data

Not a hard filter this time — report the full distribution. We'll enforce ALL laws in TLNS since the null-space ones should hold in a closed internal system, even if bulk element-balance doesn't.

In [ ]:
# For each law, compute drift across all cells
# Drift = (max - min) / |mean| across time, averaged across cells
# But actually more informative: drift per time step vs initial magnitude

drifts = np.zeros(n_laws)
for law_i in range(n_laws):
    c = C_full[law_i]
    law_values = all_data @ c  # (n_cells, n_time)
    # Per-cell relative range
    per_cell_range = (law_values.max(axis=1) - law_values.min(axis=1))
    per_cell_mean = np.abs(law_values).mean(axis=1) + 1e-9
    drifts[law_i] = np.median(per_cell_range / per_cell_mean)

print(f'Distribution of conservation-law drift across {n_laws} laws:')
for pct in [10, 25, 50, 75, 90]:
    print(f'  {pct}th percentile: {np.percentile(drifts, pct)*100:.2f}%')

# How many laws hold at various thresholds
for thresh in [0.001, 0.01, 0.05, 0.10, 0.20, 0.50]:
    n_holding = (drifts < thresh).sum()
    print(f'  Laws with drift < {thresh*100:>5.1f}%: {n_holding}/{n_laws}')

## 5. Build Δy dataset at Δt = 600 s

For each cell, extract pairs (y(t), y(t+600s)) skipping a burn-in period. Split by cell: 40 cells for training, 10 for test.

In [ ]:
DT = 600  # 10 minutes
BURN_IN = 300  # skip first 5 min of each cell
STRIDE = 60  # sample pairs every 1 minute along trajectory (avoid heavy correlation)

# Split cells
rng = np.random.default_rng(0)
cell_order = rng.permutation(N_CELLS)
train_cells = cell_order[:40]
test_cells  = cell_order[40:]
print(f'Train cells: {sorted(train_cells)}')
print(f'Test cells:  {sorted(test_cells)}')

def build_pairs(cell_indices, all_data=all_data, dt=DT, burn_in=BURN_IN, stride=STRIDE):
    n_time = all_data.shape[1]
    Y0_list, YT_list = [], []
    for c in cell_indices:
        for t in range(burn_in, n_time - dt, stride):
            Y0_list.append(all_data[c, t])
            YT_list.append(all_data[c, t + dt])
    return np.array(Y0_list), np.array(YT_list)

Y0_tr, YT_tr = build_pairs(train_cells)
Y0_te, YT_te = build_pairs(test_cells)
dY_tr = YT_tr - Y0_tr
dY_te = YT_te - Y0_te

print(f'\nTraining pairs: {Y0_tr.shape[0]:,}')
print(f'Test pairs:     {Y0_te.shape[0]:,}')
print(f'State dim:      {Y0_tr.shape[1]}')
print(f'Δy magnitude:   mean |Δy| = {np.abs(dY_tr).mean():.2f}, max = {np.abs(dY_tr).max():.0f}')

# How much do states actually change at Δt=600s?
relative_change = np.abs(dY_tr).mean(axis=0) / (np.abs(Y0_tr).mean(axis=0) + 1e-6)
print(f'\nPer-metabolite relative Δy (mean |Δy|/|y|) distribution:')
for pct in [10, 25, 50, 75, 90]:
    print(f'  {pct}th percentile: {np.percentile(relative_change, pct)*100:>6.2f}%')

## 6. Baselines + TLNS on Δy prediction

Four models:
- **Identity**: predict Δy = 0 (states don't change). Surprisingly hard baseline for stable cells.
- **Ridge**: linear regression y(t) → Δy, no conservation.
- **MLP**: 2-layer neural network y(t) → Δy, no conservation.
- **TLNS**: predict Δy in null space of C (exact conservation).

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
import time

# ---- Standardize inputs ----
x_scaler = StandardScaler()
y_scaler = StandardScaler()  # for Δy targets

Xs_tr = x_scaler.fit_transform(Y0_tr)
Xs_te = x_scaler.transform(Y0_te)
dYs_tr = y_scaler.fit_transform(dY_tr)

# ---- Model 1: Identity ----
def pred_identity(Y0):
    return np.zeros_like(Y0)  # Δy = 0

t0 = time.perf_counter()
dY_identity = pred_identity(Y0_te)
t_identity = time.perf_counter() - t0
print(f'Identity predicted in {t_identity*1e6:.0f} µs total')

# ---- Model 2: Ridge on Δy ----
from sklearn.linear_model import Ridge
t0 = time.perf_counter()
ridge = Ridge(alpha=1.0)
ridge.fit(Xs_tr, dYs_tr)
t_ridge_fit = time.perf_counter() - t0

t0 = time.perf_counter()
dYs_ridge_pred = ridge.predict(Xs_te)
t_ridge_pred = time.perf_counter() - t0
dY_ridge = y_scaler.inverse_transform(dYs_ridge_pred)
print(f'Ridge: fit in {t_ridge_fit:.1f}s, predict in {t_ridge_pred*1000:.1f}ms')

# ---- Model 3: MLP on Δy ----
from sklearn.neural_network import MLPRegressor
t0 = time.perf_counter()
mlp = MLPRegressor(hidden_layer_sizes=(128, 128), max_iter=50, random_state=0,
                    early_stopping=True, validation_fraction=0.1)
mlp.fit(Xs_tr, dYs_tr)
t_mlp_fit = time.perf_counter() - t0

t0 = time.perf_counter()
dYs_mlp_pred = mlp.predict(Xs_te)
t_mlp_pred = time.perf_counter() - t0
dY_mlp = y_scaler.inverse_transform(dYs_mlp_pred)
print(f'MLP: fit in {t_mlp_fit:.1f}s ({mlp.n_iter_} iters), predict in {t_mlp_pred*1000:.1f}ms')

# ---- Model 4: TLNS (null-space-constrained Ridge) ----
# Parametrize Δy = N @ z where C @ N = 0
# Then fit z = f(y) via ridge
# Since C_full has zero rows for unmatched metabolites, those never get
# constrained — they're fully free. That's correct.
U, s, Vt_c = np.linalg.svd(C_full, full_matrices=True)
tol = max(C_full.shape) * np.finfo(float).eps * s.max()
rank_C = int((s > tol).sum())
N = Vt_c[rank_C:].T  # null space basis, (n_met, n_met - rank_C)
print(f'\nNull space dim of C: {N.shape[1]} (C rank = {rank_C})')
err = np.linalg.norm(C_full @ N)
print(f'||C @ N|| = {err:.2e}')

# Project training Δy onto null space
z_tr = dY_tr @ N  # (n_train, d_free)
z_scaler = StandardScaler()
zs_tr = z_scaler.fit_transform(z_tr)

t0 = time.perf_counter()
tlns_ridge = Ridge(alpha=1.0)
tlns_ridge.fit(Xs_tr, zs_tr)
t_tlns_fit = time.perf_counter() - t0

t0 = time.perf_counter()
zs_tlns_pred = tlns_ridge.predict(Xs_te)
z_tlns_pred = z_scaler.inverse_transform(zs_tlns_pred)
dY_tlns = z_tlns_pred @ N.T
t_tlns_pred = time.perf_counter() - t0
print(f'TLNS: fit in {t_tlns_fit:.1f}s, predict in {t_tlns_pred*1000:.1f}ms')

## 7. Results — R² per metabolite

In [ ]:
def r2_per_metabolite(dY_pred, dY_true):
    """R² on Δy per metabolite, using only metabolites that actually change."""
    r2s = np.full(dY_true.shape[1], np.nan)
    for j in range(dY_true.shape[1]):
        v = dY_true[:, j].var()
        if v > 1e-6:  # only score metabolites that actually change
            r2s[j] = r2_score(dY_true[:, j], dY_pred[:, j])
    return r2s

r2_identity = r2_per_metabolite(dY_identity, dY_te)
r2_ridge = r2_per_metabolite(dY_ridge, dY_te)
r2_mlp = r2_per_metabolite(dY_mlp, dY_te)
r2_tlns = r2_per_metabolite(dY_tlns, dY_te)

n_scorable = (~np.isnan(r2_ridge)).sum()
print(f'Metabolites that change at Δt=600s: {n_scorable}/{len(MET_IDS)}\n')

print(f'{"Model":<12s} {"R² mean":>9s} {"R² median":>10s} {"R²>0 %":>8s} {"R²>0.3 %":>10s}')
for name, r2 in [('Identity', r2_identity), ('Ridge', r2_ridge), ('MLP', r2_mlp), ('TLNS', r2_tlns)]:
    valid = ~np.isnan(r2)
    if valid.sum() == 0:
        print(f'  {name:<12s} no valid R²')
        continue
    vals = r2[valid]
    mean, median = np.nanmean(vals), np.nanmedian(vals)
    frac_pos = (vals > 0).mean() * 100
    frac_03 = (vals > 0.3).mean() * 100
    print(f'  {name:<12s} {mean:>+9.3f} {median:>+10.3f} {frac_pos:>7.1f}% {frac_03:>9.1f}%')

## 8. Results — conservation violations

In [ ]:
def conservation_violations(dY_pred, C=C_full):
    """|C @ Δy_pred| per sample, averaged. Should be 0 if Δy lies in null space."""
    # Only laws that actually hold in real data (drift < 5%)
    real_laws = drifts < 0.05
    C_real = C[real_laws]
    if C_real.shape[0] == 0:
        return np.nan, 0
    violations = np.abs(dY_pred @ C_real.T)  # (n_samples, n_real_laws)
    return violations.mean(), C_real.shape[0]

print('Conservation violations on held-out test set:')
print('(Mean |C @ Δy| per sample; lower = better)')
print(f'(Using only laws with <5% drift in training data)\n')

for name, dY in [('Identity', dY_identity), ('Ridge', dY_ridge),
                  ('MLP', dY_mlp), ('TLNS', dY_tlns)]:
    viol, n_laws_used = conservation_violations(dY)
    print(f'  {name:<12s} mean violation = {viol:.3e}  ({n_laws_used} laws enforced)')

# Compute improvement ratios
ridge_viol, _ = conservation_violations(dY_ridge)
tlns_viol, _ = conservation_violations(dY_tlns)
if tlns_viol > 1e-20:
    ratio = ridge_viol / tlns_viol
    print(f'\nRidge/TLNS violation ratio: {ratio:.2e}×')
elif ridge_viol > 0:
    print(f'\nTLNS violation: effectively zero ({tlns_viol:.2e}), Ridge: {ridge_viol:.3e}')
    print(f'Lower bound on ratio: {ridge_viol/1e-15:.2e}×')

## 9. Runtime comparison

In [ ]:
# Per-query speed
n_queries = Y0_te.shape[0]
print(f'Per-query inference time (batch of {n_queries}):')
print(f'  Identity:  {t_identity*1e6/n_queries:>7.2f} µs/query')
print(f'  Ridge:     {t_ridge_pred*1e6/n_queries:>7.2f} µs/query')
print(f'  MLP:       {t_mlp_pred*1e6/n_queries:>7.2f} µs/query')
print(f'  TLNS:      {t_tlns_pred*1e6/n_queries:>7.2f} µs/query')

# For context: Thornburg 4DWCM takes ~6 days per cell cycle on 2 GPUs
# = 6 * 86400 seconds / 7200 time points / 2 GPUs = 36 GPU-seconds per time step
# ≈ 36e6 µs per trajectory step, per GPU
# Our surrogate does the same thing in ~microseconds per step
thornburg_per_step_us = 6 * 86400 * 1e6 / 7200 / 2
tlns_per_step_us = t_tlns_pred * 1e6 / n_queries
print(f'\nContext: Thornburg 4DWCM full simulation ≈ {thornburg_per_step_us:.0f} µs per time step per GPU')
print(f'TLNS speedup over mechanistic sim: {thornburg_per_step_us/tlns_per_step_us:.0e}×')
print('(Caveat: TLNS predicts only metabolites, Thornburg simulates everything)')

## 10. Success criteria check and save

In [ ]:
import json

results = {
    'n_cells_total': N_CELLS,
    'n_train_cells': int(len(train_cells)),
    'n_test_cells': int(len(test_cells)),
    'n_train_pairs': int(Y0_tr.shape[0]),
    'n_test_pairs': int(Y0_te.shape[0]),
    'n_cytoplasm_metabolites': int(len(MET_IDS)),
    'n_imb155_matched': len(thornburg_in_imb),
    'n_null_space_laws': int(n_laws),
    'n_laws_holding_5pct': int((drifts < 0.05).sum()),
    'n_laws_holding_1pct': int((drifts < 0.01).sum()),
    'dt_seconds': DT,
    'n_scorable_metabolites': int(n_scorable),
}

for name, r2, dY in [('identity', r2_identity, dY_identity),
                      ('ridge',    r2_ridge,    dY_ridge),
                      ('mlp',      r2_mlp,      dY_mlp),
                      ('tlns',     r2_tlns,     dY_tlns)]:
    valid = ~np.isnan(r2)
    viol, _ = conservation_violations(dY)
    results[name] = {
        'r2_mean': float(np.nanmean(r2)) if valid.any() else None,
        'r2_median': float(np.nanmedian(r2)) if valid.any() else None,
        'r2_gt_0_frac': float((r2[valid] > 0).mean()) if valid.any() else None,
        'r2_gt_0_3_frac': float((r2[valid] > 0.3).mean()) if valid.any() else None,
        'conservation_violation': float(viol) if not np.isnan(viol) else None,
    }

results['timing_us_per_query'] = {
    'identity': float(t_identity * 1e6 / n_queries),
    'ridge':    float(t_ridge_pred * 1e6 / n_queries),
    'mlp':      float(t_mlp_pred * 1e6 / n_queries),
    'tlns':     float(t_tlns_pred * 1e6 / n_queries),
}

# Success criteria
tlns_r2_median = results['tlns']['r2_median'] or -np.inf
ridge_viol = results['ridge']['conservation_violation'] or np.inf
tlns_viol = results['tlns']['conservation_violation'] or np.inf
viol_ratio = ridge_viol / max(tlns_viol, 1e-20)
speedup = thornburg_per_step_us / tlns_per_step_us

criteria = {
    'r2_above_0_3': tlns_r2_median > 0.3,
    'conservation_100x_better': viol_ratio > 100,
    'speedup_1000x': speedup > 1000,
}
results['success_criteria'] = criteria
n_met = sum(criteria.values())

print(f'\nSuccess criteria:')
print(f'  [{chr(0x2713) if criteria["r2_above_0_3"] else chr(0x2717)}] TLNS R² median > 0.3:        {tlns_r2_median:+.3f}')
print(f'  [{chr(0x2713) if criteria["conservation_100x_better"] else chr(0x2717)}] Conservation 100× better:    {viol_ratio:.2e}×')
print(f'  [{chr(0x2713) if criteria["speedup_1000x"] else chr(0x2717)}] Speedup > 1000×:             {speedup:.2e}×')
print(f'\n  {n_met}/3 criteria met')

# Save
out_path = '/content/drive/MyDrive/tlns_proper_results.json'
with open(out_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f'\nSaved: {out_path}')
print(json.dumps(results, indent=2))